# 10_build_and_run

10_build_and_run.py — 자기교정형 Agentic RAG 그래프 조립 + 실행 (메인)

흐름:
  retrieve → grade_documents
            ├ (관련 있음) → generate → (충실성/유용성)
            │                            ├ 통과 → END
            │                            ├ 환각 → generate 재시도
            │                            └ 미흡 → transform_query → web_search → grade_documents (순환)
            └ (관련 없음) → transform_query → web_search → grade_documents (순환)

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
import nest_asyncio; nest_asyncio.apply()
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '10_build_and_run.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
10_build_and_run.py — 자기교정형 Agentic RAG 그래프 조립 + 실행 (메인)

흐름:
  retrieve → grade_documents
            ├ (관련 있음) → generate → (충실성/유용성)
            │                            ├ 통과 → END
            │                            ├ 환각 → generate 재시도
            │                            └ 미흡 → transform_query → web_search → grade_documents (순환)
            └ (관련 없음) → transform_query → web_search → grade_documents (순환)
"""
import importlib
import sys
from pathlib import Path
from typing import List, TypedDict

from langgraph.graph import StateGraph, START, END

# 숫자로 시작하는 08_nodes.py / 09_conditional_edges.py 는 일반 import 불가 → importlib 로 우회
sys.path.insert(0, str(Path(__file__).resolve().parent))
_nodes = importlib.import_module("08_nodes")
_edges = importlib.import_module("09_conditional_edges")


class GraphState(TypedDict):
    question: str
    documents: List[str]
    generation: str
    retry_count: int
    web_search: str


def build_app():
    workflow = StateGraph(GraphState)

    # 노드 등록
    workflow.add_node("retrieve",        _nodes.retrieve)
    workflow.add_node("grade_documents", _nodes.grade_documents)
    workflow.add_node("generate",        _nodes.generate)
    workflow.add_node("transform_query", _nodes.transform_query)
    workflow.add_node("do_web_search",   _nodes.web_search)  # LangGraph 1.x: 노드명 ≠ State 필드명

    # 엣지 연결
    workflow.add_edge(START, "retrieve")
    workflow.add_edge("retrieve", "grade_documents")

    workflow.add_conditional_edges(
        "grade_documents",
        _edges.decide_to_generate,
        {"transform_query": "transform_query", "generate": "generate"},
    )
    workflow.add_edge("transform_query", "do_web_search")
    workflow.add_edge("do_web_search", "grade_documents")   # 순환!

    workflow.add_conditional_edges(
        "generate",
        _edges.grade_generation,
        {
            "not_supported": "generate",     # 환각 → 재생성
            "useful":         END,           # 통과 → 종료
            "not_useful":     "transform_query",
        },
    )

    return workflow.compile()


def run_one(question: str) -> None:
    app = build_app()
    inputs = {"question": question, "retry_count": 0}

    print("\n" + "═" * 70)
    print(f"🚀 RUN: {question}")
    print("═" * 70)

    last_state = None
    for output in app.stream(inputs, {"recursion_limit": 25}):
        for node_name, payload in output.items():
            print(f"  ▶ [{node_name}] 완료")
            last_state = payload

    if last_state and "generation" in last_state:
        print("\n📝 최종 답변")
        print("-" * 70)
        print(last_state["generation"])
    else:
        print("\n(생성 단계 도달 못함)")


if __name__ == "__main__":
    # ── 데모 1: SAMPLE_DOCS 에 답이 있는 케이스 (벡터DB 경로만 사용) ──
    #    예상 트레이스: retrieve → grade(통과 ≥1) → generate → useful=yes → END
    run_one("에이전트 메모리에는 어떤 종류가 있나?")

    # ── 데모 2: SAMPLE_DOCS 에 답이 없는 케이스 (웹 폴백 경로) ──
    #    예상 트레이스: retrieve → grade(0/4) → transform_query → web_search →
    #                  grade(다시) → generate → END (혹은 재시도 후 종료)
    #    핵심 관찰 포인트:
    #      - decide_to_generate 가 "transform_query" 로 분기
    #      - web_search → grade_documents 의 *순환 (cycle)* 엣지가 발화
    #      - retry_count 가 MAX_RETRIES 에 도달하면 강제 종료되는 안전장치
    run_one("오늘 비트코인 가격은 얼마인가?")

D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



══════════════════════════════════════════════════════════════════════
🚀 RUN: 에이전트 메모리에는 어떤 종류가 있나?
══════════════════════════════════════════════════════════════════════


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6596.69it/s]

--- RETRIEVE ---
  ▶ [retrieve] 완료
--- GRADE DOCUMENTS ---


...

    재작성: 비트코인 현재 가격 조회
  ▶ [transform_query] 완료
--- WEB SEARCH (DuckDuckGo) ---
  ▶ [do_web_search] 완료
--- GRADE DOCUMENTS ---


D:\git\2604_agent_210h_handson\supp\_common.py:243: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


    0/1 통과 → web_search=Yes
--- ASSESS GRADED DOCUMENTS ---
    재시도 상한 도달 → 강제 generate
  ▶ [grade_documents] 완료
--- GENERATE ---


--- CHECK HALLUCINATION & USEFULNESS ---


    환각 감지 but 상한 도달 → useful 로 종료
  ▶ [generate] 완료

📝 최종 답변
----------------------------------------------------------------------
비트코인의 실시간 가격은 제가 알 수 없습니다. 실시간 정보가 필요하시면 인터넷 검색이나 암호화폐 거래소 앱을 확인해 주세요.
